# XGBoost Model — Seasonal NYC Airbnb Price Prediction

**Playground: seasonal_approach**

Trains an XGBoost regressor on the stacked monthly feature matrix produced by `02_stacking.ipynb`.

**Inputs (read-only):**
- `playground/seasonal_approach/data/stacked_features.csv` — 28,984 rows × 64 cols (4,073 listings × 8 months)
- `playground/seasonal_approach/data/seasonal_index.csv` — review-volume seasonal index by month

**Key design choices:**
- Target: `log_price` (log₁+₁ of nightly price — normalises right-skewed distribution)
- Train/test split: months 4–10 as train, month 11 (November) as held-out test
- `month` and `seasonal_index` are both model features (not only used at inference)
- Evaluation on original dollar scale via `expm1(prediction)`

**Writes to (playground only):**
- `playground/seasonal_approach/data/xgb_model.json` — serialised model
- `playground/seasonal_approach/data/eval_metrics.csv` — per-fold + test metrics
- `playground/seasonal_approach/data/shap_summary.csv` — mean |SHAP| per feature


In [1]:
import warnings, json
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')

DATA = Path('data')
STACKED  = DATA / 'stacked_features.csv'
SEA_IDX  = DATA / 'seasonal_index.csv'
MODEL_OUT = DATA / 'xgb_model.json'
EVAL_OUT  = DATA / 'eval_metrics.csv'
SHAP_OUT  = DATA / 'shap_summary.csv'

for p in [STACKED, SEA_IDX]:
    print(f'{"✓" if p.exists() else "✗ MISSING"}  {p.name}')


✓  stacked_features.csv
✓  seasonal_index.csv


---
## 1. Load data and define features


In [2]:
df = pd.read_csv(STACKED)
print(f'Stacked data: {df.shape}')
print(f'Months: {sorted(df["month"].unique())}')
print(f'Listings: {df["id"].nunique():,}')
print(f'Nulls: {df.isnull().sum().sum()}')


Stacked data: (28244, 64)
Months: [np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11)]
Listings: 4,014
Nulls: 0


In [3]:
# Feature columns = everything except metadata and target
NON_FEATURES = {'id', 'price_numeric', 'log_price'}

# --- LEAKAGE REMOVAL ---
# InsideAirbnb computes:
#   estimated_revenue_l365d  = price × estimated_occupied_nights
#   estimated_occupancy_l365d = estimated_occupied_nights
# So: price = estimated_revenue / estimated_occupancy  (exact algebraic identity).
# Including either feature would give the model the answer.
# Verified: (revenue/occupancy) == price for 100% of rows within $1.
LEAK_FEATURES = {'estimated_revenue_l365d', 'estimated_occupancy_l365d'}

FEATURE_COLS = [c for c in df.columns if c not in NON_FEATURES and c not in LEAK_FEATURES]

print(f'Feature columns ({len(FEATURE_COLS)}) — leaking features removed:')
for i, c in enumerate(FEATURE_COLS, 1):
    print(f'  {i:2d}. {c}')

Feature columns (59) — leaking features removed:
   1. accommodates
   2. bedrooms
   3. beds
   4. bathrooms
   5. room_type_ord
   6. amenity_count
   7. minimum_nights
   8. availability_365
   9. number_of_reviews
  10. number_of_reviews_ltm
  11. reviews_per_month
  12. review_scores_rating
  13. review_scores_accuracy
  14. review_scores_cleanliness
  15. review_scores_checkin
  16. review_scores_communication
  17. review_scores_location
  18. review_scores_value
  19. is_superhost
  20. host_identity_verified_f
  21. is_instant_bookable
  22. host_response_rate_f
  23. host_acceptance_rate_f
  24. host_response_time_ord
  25. has_license
  26. has_wifi
  27. has_kitchen
  28. has_ac
  29. has_gym
  30. has_parking
  31. has_pool
  32. has_washer
  33. has_dryer
  34. has_elevator
  35. has_doorman
  36. has_pets_allowed
  37. dist_subway_km
  38. near_subway
  39. subway_lines_500m
  40. nearest_subway_cbd
  41. nearest_subway_ada
  42. dist_lirr_km
  43. lirr_count_1mi
  44. p

---
## 2. Train / test split

**Strategy:** months 4–10 → train (24,911 rows), month 11 → held-out test (4,073 rows).

Month 11 (November) is chosen as the test set because:
- It has the highest coverage (all 4,073 STR listings reported a price)
- It represents the baseline month for which we have the most ground-truth signal
- Keeping it held out gives an unbiased final evaluation


In [4]:
train = df[df['month'] != 11].copy()
test  = df[df['month'] == 11].copy()

X_train = train[FEATURE_COLS]
y_train = train['log_price']

X_test  = test[FEATURE_COLS]
y_test  = test['log_price']

print(f'Train: {X_train.shape}  ({train["month"].nunique()} months)')
print(f'Test:  {X_test.shape}   (November only)')
print(f'Train price median: ${np.expm1(y_train.median()):.0f}')
print(f'Test  price median: ${np.expm1(y_test.median()):.0f}')


Train: (24338, 59)  (7 months)
Test:  (3906, 59)   (November only)
Train price median: $197
Test  price median: $214


---
## 3. XGBoost hyperparameters

Conservative defaults — tuned for M2 MacBook Air (no GPU).
A subsequent grid search / Optuna sweep can improve these further.


In [5]:
PARAMS = dict(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    min_child_weight  = 5,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,      # L1
    reg_lambda        = 1.0,      # L2
    objective         = 'reg:squarederror',
    eval_metric       = 'rmse',
    tree_method       = 'hist',   # fastest on M2
    device            = 'cpu',
    random_state      = 42,
    early_stopping_rounds = 30,
    verbosity         = 0,
)
print('Parameters set:', {k: v for k, v in PARAMS.items() if k not in ('objective','eval_metric','tree_method','device','verbosity')})


Parameters set: {'n_estimators': 500, 'learning_rate': 0.05, 'max_depth': 6, 'min_child_weight': 5, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.1, 'reg_lambda': 1.0, 'random_state': 42, 'early_stopping_rounds': 30}


---
## 4. 5-fold cross-validation on training set

Cross-validate on the training portion (Apr–Oct) to get stable in-sample estimates
before evaluating on the held-out November test set.


In [6]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train), 1):
    Xtr, Xval = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    ytr, yval = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    model = xgb.XGBRegressor(**PARAMS)
    model.fit(
        Xtr, ytr,
        eval_set=[(Xval, yval)],
        verbose=False
    )

    pred_log = model.predict(Xval)
    pred_price = np.expm1(pred_log)
    true_price = np.expm1(yval.values)

    rmse_log   = np.sqrt(mean_squared_error(yval, pred_log))
    mae_price  = mean_absolute_error(true_price, pred_price)
    r2         = r2_score(yval, pred_log)
    mape       = np.mean(np.abs((true_price - pred_price) / true_price)) * 100

    cv_results.append({
        'fold': fold, 'n_trees': model.best_iteration,
        'rmse_log': round(rmse_log, 4),
        'mae_price': round(mae_price, 2),
        'r2': round(r2, 4),
        'mape_pct': round(mape, 2),
    })
    print(f'  Fold {fold}: RMSE(log)={rmse_log:.4f}  MAE($)={mae_price:.1f}  R²={r2:.4f}  MAPE={mape:.1f}%  trees={model.best_iteration}')

cv_df = pd.DataFrame(cv_results)
print(f'\n5-fold mean ± std:')
for col in ['rmse_log', 'mae_price', 'r2', 'mape_pct']:
    print(f'  {col:12s}: {cv_df[col].mean():.4f} ± {cv_df[col].std():.4f}')


  Fold 1: RMSE(log)=0.1722  MAE($)=35.7  R²=0.9307  MAPE=12.7%  trees=499


  Fold 2: RMSE(log)=0.1715  MAE($)=35.3  R²=0.9300  MAPE=12.5%  trees=499


  Fold 3: RMSE(log)=0.1727  MAE($)=35.8  R²=0.9314  MAPE=12.5%  trees=499


  Fold 4: RMSE(log)=0.1820  MAE($)=37.8  R²=0.9228  MAPE=12.8%  trees=499


  Fold 5: RMSE(log)=0.1757  MAE($)=36.3  R²=0.9305  MAPE=12.5%  trees=499

5-fold mean ± std:
  rmse_log    : 0.1748 ± 0.0043
  mae_price   : 36.1940 ± 0.9790
  r2          : 0.9291 ± 0.0035
  mape_pct    : 12.6080 ± 0.1226


---
## 5. Final model — train on all 7 months, evaluate on November


In [7]:
# Use median best_iteration from CV folds as fixed n_estimators (no early stopping on full train)
best_n = int(cv_df['n_trees'].median())
FINAL_PARAMS = {k: v for k, v in PARAMS.items()
                if k not in ('early_stopping_rounds',)}
FINAL_PARAMS['n_estimators'] = best_n

final_model = xgb.XGBRegressor(**FINAL_PARAMS)
final_model.fit(X_train, y_train, verbose=False)
print(f'Final model trained with {best_n} trees on {len(X_train):,} rows')

# Evaluate on held-out November
pred_log   = final_model.predict(X_test)
pred_price = np.expm1(pred_log)
true_price = np.expm1(y_test.values)

test_rmse_log  = np.sqrt(mean_squared_error(y_test, pred_log))
test_mae_price = mean_absolute_error(true_price, pred_price)
test_r2        = r2_score(y_test, pred_log)
test_mape      = np.mean(np.abs((true_price - pred_price) / true_price)) * 100
test_med_err   = np.median(np.abs(true_price - pred_price))

print(f'\n=== HELD-OUT TEST (November 2025) ===')
print(f'  RMSE (log scale):  {test_rmse_log:.4f}')
print(f'  MAE  ($ scale):    ${test_mae_price:.2f}')
print(f'  MedAE ($ scale):   ${test_med_err:.2f}')
print(f'  R²   (log scale):  {test_r2:.4f}')
print(f'  MAPE:              {test_mape:.1f}%')
print(f'\nBaseline (predict median always):')
med_pred = np.median(true_price)
baseline_mae = mean_absolute_error(true_price, np.full_like(true_price, med_pred))
print(f'  Median-baseline MAE: ${baseline_mae:.2f}')
print(f'  Improvement over baseline: {(1 - test_mae_price/baseline_mae)*100:.1f}%')


Final model trained with 499 trees on 24,338 rows

=== HELD-OUT TEST (November 2025) ===
  RMSE (log scale):  0.1966
  MAE  ($ scale):    $45.35
  MedAE ($ scale):   $20.28
  R²   (log scale):  0.9154
  MAPE:              13.6%

Baseline (predict median always):
  Median-baseline MAE: $153.45
  Improvement over baseline: 70.4%


---
## 6. Residual analysis


In [8]:
residuals = true_price - pred_price
pct_within_20 = np.mean(np.abs(residuals) <= 0.20 * true_price) * 100
pct_within_50 = np.mean(np.abs(residuals) <= 50)                * 100

print(f'Predictions within 20% of true price:  {pct_within_20:.1f}%')
print(f'Predictions within $50 of true price:  {pct_within_50:.1f}%')
print()

# Price-bucket breakdown
test_df = test[['price_numeric']].copy()
test_df['pred_price'] = pred_price
test_df['abs_err']    = np.abs(test_df['pred_price'] - test_df['price_numeric'])
test_df['bucket'] = pd.cut(
    test_df['price_numeric'],
    bins=[0, 75, 150, 250, 500, 10000],
    labels=['<$75', '$75-150', '$150-250', '$250-500', '$500+']
)

bucket_stats = test_df.groupby('bucket', observed=True).agg(
    count=('abs_err', 'count'),
    mae=('abs_err', 'mean'),
    mape=('price_numeric', lambda x: (test_df.loc[x.index, 'abs_err'] / x).mean() * 100)
).round(1)
print('Error by price bucket:')
print(bucket_stats.to_string())


Predictions within 20% of true price:  77.4%
Predictions within $50 of true price:  74.4%

Error by price bucket:
          count    mae  mape
bucket                      
<$75         93   10.3  17.1
$75-150    1156   12.5  11.0
$150-250   1024   24.0  12.0
$250-500   1127   55.6  15.5
$500+       506  147.3  18.1


---
## 7. SHAP feature importance

SHAP (SHapley Additive exPlanations) gives each feature an additive contribution
to each individual prediction. `mean |SHAP|` across all test samples is the standard
global importance metric — unlike XGBoost's built-in `gain`, it accounts for feature
interactions and is model-agnostic.


In [9]:
print('Computing SHAP values on test set...')
explainer   = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_test)

mean_abs_shap = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=FEATURE_COLS
).sort_values(ascending=False)

print('\nTop 20 features by mean |SHAP|:')
print(mean_abs_shap.head(20).round(4).to_string())


Computing SHAP values on test set...



Top 20 features by mean |SHAP|:
prop_Private room            0.2221
accommodates                 0.1161
boro_Manhattan               0.0815
number_of_reviews_ltm        0.0809
bedrooms                     0.0695
review_scores_location       0.0561
beds                         0.0494
poi_count_1km                0.0479
month                        0.0457
review_scores_rating         0.0441
review_scores_value          0.0435
dist_lirr_km                 0.0357
review_scores_cleanliness    0.0341
amenity_count                0.0325
has_elevator                 0.0298
nearest_subway_cbd           0.0286
seasonal_index               0.0286
availability_365             0.0259
bathrooms                    0.0256
has_washer                   0.0215


---
## 8. Save outputs


In [10]:
# Model
final_model.save_model(str(MODEL_OUT))
print(f'Model saved: {MODEL_OUT}')

# Eval metrics
all_metrics = cv_df.copy()
all_metrics['split'] = 'cv_fold_' + all_metrics['fold'].astype(str)

test_row = pd.DataFrame([{
    'fold': 0, 'split': 'test_november',
    'n_trees': best_n,
    'rmse_log': round(test_rmse_log, 4),
    'mae_price': round(test_mae_price, 2),
    'r2': round(test_r2, 4),
    'mape_pct': round(test_mape, 2),
}])
all_metrics = pd.concat([all_metrics, test_row], ignore_index=True)
all_metrics.to_csv(EVAL_OUT, index=False)
print(f'Metrics saved: {EVAL_OUT}')

# SHAP summary
shap_df = mean_abs_shap.reset_index()
shap_df.columns = ['feature', 'mean_abs_shap']
shap_df.to_csv(SHAP_OUT, index=False)
print(f'SHAP saved:   {SHAP_OUT}')

print('\n=== FINAL SUMMARY ===')
print(f'Training rows:        {len(X_train):,}  (Apr–Oct, 7 months)')
print(f'Test rows:            {len(X_test):,}   (November)')
print(f'Features:             {len(FEATURE_COLS)}')
print(f'Trees (final model):  {best_n}')
print(f'CV mean R²:           {cv_df["r2"].mean():.4f}')
print(f'CV mean MAE ($):      ${cv_df["mae_price"].mean():.2f}')
print(f'Test R²:              {test_r2:.4f}')
print(f'Test MAE ($):         ${test_mae_price:.2f}')
print(f'Test MedAE ($):       ${test_med_err:.2f}')
print(f'Test MAPE:            {test_mape:.1f}%')


Model saved: data/xgb_model.json
Metrics saved: data/eval_metrics.csv
SHAP saved:   data/shap_summary.csv

=== FINAL SUMMARY ===
Training rows:        24,338  (Apr–Oct, 7 months)
Test rows:            3,906   (November)
Features:             59
Trees (final model):  499
CV mean R²:           0.9291
CV mean MAE ($):      $36.19
Test R²:              0.9154
Test MAE ($):         $45.35
Test MedAE ($):       $20.28
Test MAPE:            13.6%
